# Module 1 — Earth Engine Fundamentals

**Goal:** understand what Earth Engine actually is, get authenticated from Python, and learn the two core data types you'll use in every future project: `ee.Image` and `ee.ImageCollection`.

**Time:** ~1.5–2 hours

### Why this matters for the glyphosate project
Everything in that pipeline — vegetation index time series, COMTRS zonal stats, harmonic baselines — is built on top of `Image`, `ImageCollection`, and `Feature`/`FeatureCollection` objects. If those four types feel natural, the rest of GEE is just combining them in different ways.

### The mental model
Earth Engine is **not** a Python library that downloads pixels to your machine. It's a client that builds a description of a computation (a graph of operations) and ships that description to Google's servers, which run it on petabytes of imagery and send back only the small result you asked for (a chart, a table, a small export). This is why:
- You can't `print()` an `ee.Image` and see pixel values — it's a *reference* to a server-side object.
- Computations are lazy until you call something that forces evaluation (`.getInfo()`, `.getThumbURL()`, an `Export`, etc.)
- The skill to learn isn't "Python image processing" — it's "how do I describe my computation so the server does the heavy lifting."


In [ ]:
# --- Setup ---
# Run this once. It opens a browser flow to authenticate your Google account
# to Earth Engine. If you don't have access yet, sign up at
# https://code.earthengine.google.com/register

import ee

try:
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')  # replace with your GEE cloud project
except Exception:
    ee.Authenticate()
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')

print("Earth Engine initialized OK")


### `ee.Image` — a single raster

An `ee.Image` is a stack of bands. Each band has a name, a data type, a resolution (scale), and a projection. Let's load a real Landsat scene and inspect it *without* downloading any pixels.

In [ ]:
img = ee.Image('LANDSAT/LC08/C02/T1_L2/LC08_044034_20200827') #not a real image, pick one you like!

# .getInfo() is the moment you cross from "server-side description" to
# "actual Python dict on your machine." Use it sparingly — only for metadata,
# never to pull down full-resolution pixel grids.
info = img.getInfo()
print(list(info.keys()))
print([b['id'] for b in info['bands']])


In [ ]:
# Band math happens server-side, symbolically, until evaluated.
nir = img.select('SR_B5')
red = img.select('SR_B4')

ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')
print(ndvi)  # <-- this is a description of a computation, not pixel values


Notice that `print(ndvi)` shows you an `ee.Image` object repr, not numbers. That's expected — nothing has run on the server yet.

### Exercise 1.1
Compute NDVI for the image above using the built-in `.normalizedDifference()` helper (it does the subtract/divide/rename for you) and confirm it matches your manual version at one point using `.getInfo()` on a `.reduceRegion()` call (we'll cover reducers properly in Module 4 — for now just try it and see what comes back).

In [ ]:
# TODO: your code here
# hint: img.normalizedDifference(['SR_B5', 'SR_B4'])


### `ee.ImageCollection` — a stack of images over time

This is the object you'll live in for the glyphosate project: a time series of scenes over California, filtered by date and location.

In [ ]:
collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
              .filterDate('2020-06-01', '2020-09-01')
              .filterBounds(ee.Geometry.Point(-121.5, 38.5)))

print('Number of images:', collection.size().getInfo())


`.size()` is another server-side object (`ee.Number`) — you still needed `.getInfo()` to pull down the actual integer.

### Exercise 1.2
1. Build an `ImageCollection` for `'COPERNICUS/S2_SR_HARMONIZED'` (Sentinel-2) filtered to a 2-month window and a point somewhere in California's Central Valley.
2. Print how many images it contains.
3. Grab the **first** image (`.first()`) and print its band names.


In [ ]:
# TODO: your code here


### Project 1 — "Metadata explorer"

Write a small function `describe_collection(collection_id, start, end, lon, lat)` that:
1. Builds the filtered `ImageCollection`
2. Returns a dict with: image count, list of band names (from the first image), and the date of the first and last image in the collection

This is the kind of quick sanity-check tool you'll reach for constantly once you're building a real pipeline — before running anything expensive you want to know "how much data am I actually working with here."

Hint: to get dates as strings, use `.get('system:time_start')` on an image, wrapped in `ee.Date(...).format()`.

In [ ]:
def describe_collection(collection_id, start, end, lon, lat):
    # TODO
    pass

# describe_collection('LANDSAT/LC08/C02/T1_L2', '2020-01-01', '2020-12-31', -121.5, 38.5)


---
**Checkpoint:** you should now be able to explain, in your own words, the difference between a value that lives client-side (in Python, a real number/list/dict) and one that lives server-side (an `ee` object that's just a computation graph). If that distinction is fuzzy, re-read the mental model section before moving to Module 2 — it's the single idea that makes everything else in GEE click.